# Floor2Model — Phase 2 Training (Kaggle)
**Run cells one by one. Wait for each to finish (✓) before running the next.**

Before running:
- Settings → Accelerator → GPU T4 x2
- Settings → Internet → ON

In [ ]:
# ── Cell 1: Verify GPU & install packages ────────────────────────────
# Expected output: CUDA: True, GPU: Tesla T4
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU found! Go to Settings → Accelerator → GPU T4 x2')

!pip install ultralytics zenodo-get -q
print('\nPackages installed OK')

In [ ]:
# ── Cell 2: Clone your GitHub repo ───────────────────────────────────
# Expected output: cloning messages, then list of project files
import os

REPO_URL = 'https://github.com/Tanishqueue/floor2model.git'
WORK_DIR = '/kaggle/working'
REPO_DIR = f'{WORK_DIR}/floor2model'

if os.path.exists(REPO_DIR):
    print('Repo already exists — pulling latest...')
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'\nWorking directory: {os.getcwd()}')
!ls

In [ ]:
# ── Cell 3: Download CubiCasa5k dataset ──────────────────────────────
# Expected output: SUCCESS: All specified files have been processed
# This takes ~5-8 minutes (5.5 GB)
import os

DATASET_DIR  = '/kaggle/working/data/cubicasa5k'
DATASET_ZIP  = f'{DATASET_DIR}/cubicasa5k.zip'

os.makedirs(DATASET_DIR, exist_ok=True)

if os.path.exists(DATASET_ZIP):
    print('Zip already downloaded — skipping download')
else:
    print('Downloading CubiCasa5k (~5.5 GB) — please wait...')
    !zenodo_get 2613548 -o {DATASET_DIR}

print(f'\nContents of {DATASET_DIR}:')
!ls -lh {DATASET_DIR}

In [ ]:
# ── Cell 4: Unzip dataset ─────────────────────────────────────────────
# Expected output: Unzip complete! and folder list including high_quality/
# This takes ~3-4 minutes
import zipfile, os

DATASET_DIR = '/kaggle/working/data/cubicasa5k'
DATASET_ZIP = f'{DATASET_DIR}/cubicasa5k.zip'

# Check if already unzipped
already_extracted = any(
    os.path.isdir(os.path.join(DATASET_DIR, d))
    for d in os.listdir(DATASET_DIR)
    if d != '__MACOSX'
)

if already_extracted:
    print('Already unzipped — skipping')
else:
    print('Unzipping dataset — please wait (~3-4 mins)...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(DATASET_DIR)
    print('Unzip complete!')

print(f'\nDataset contents:')
for item in os.listdir(DATASET_DIR):
    print(f'  {item}')

In [ ]:
# ── Cell 5: Convert dataset to YOLO format ────────────────────────────
# Expected output: train/val/test split sizes, then 'Dataset ready'
# This takes ~5-10 minutes
import sys
sys.path.insert(0, '/kaggle/working/floor2model')

from src.segmentation.dataset import CubiCasaDataset

CUBICASA_DIR = '/kaggle/working/data/cubicasa5k'
YOLO_DIR     = '/kaggle/working/data/yolo_dataset'

ds = CubiCasaDataset(CUBICASA_DIR)
yaml_path = ds.prepare(output_dir=YOLO_DIR)

print(f'\nDataset ready!')
print(f'Config: {yaml_path}')

# Verify structure
import os
for split in ['train', 'val', 'test']:
    img_count = len(os.listdir(f'{YOLO_DIR}/images/{split}'))
    print(f'  {split}: {img_count} images')

In [ ]:
# ── Cell 6: Train YOLOv8s-seg for 100 epochs ─────────────────────────
# Expected duration: ~2.5 hours on T4
# Watch for mAP50 improving each epoch in the output
from ultralytics import YOLO

YOLO_YAML    = '/kaggle/working/data/yolo_dataset/dataset.yaml'
MODELS_DIR   = '/kaggle/working/models'

# Load pretrained YOLOv8s-seg (downloads automatically ~22 MB)
model = YOLO('yolov8s-seg.pt')

print('Starting training — 100 epochs, T4 GPU')
print('Expected time: ~2.5 hours\n')

results = model.train(
    data=YOLO_YAML,
    epochs=100,
    batch=16,
    imgsz=640,
    device='0',
    project=MODELS_DIR,
    name='floor_plan_seg',
    # Augmentation
    augment=True,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    # Checkpointing
    save=True,
    save_period=10,
    patience=30,
    plots=True,
    verbose=True,
)

BEST_PT = f'{results.save_dir}/weights/best.pt'
print(f'\nTraining complete!')
print(f'Best weights saved at: {BEST_PT}')

In [ ]:
# ── Cell 7: Evaluate on test set ──────────────────────────────────────
# Expected: mAP50 ~0.72-0.78 for structural elements
from ultralytics import YOLO

BEST_PT   = f'{results.save_dir}/weights/best.pt'
YOLO_YAML = '/kaggle/working/data/yolo_dataset/dataset.yaml'

model = YOLO(BEST_PT)
metrics = model.val(
    data=YOLO_YAML,
    split='test',
    device='0',
)

print('\n── Evaluation Results ──')
print(f'mAP50    : {metrics.seg.map50:.4f}')
print(f'mAP50-95 : {metrics.seg.map:.4f}')
print('\nGood targets:')
print('  mAP50 > 0.70 = solid model, ready for Phase 3')
print('  mAP50 > 0.78 = excellent')

In [ ]:
# ── Cell 8: Quick visual test on one floor plan ───────────────────────
import sys, os, cv2
import matplotlib.pyplot as plt
sys.path.insert(0, '/kaggle/working/floor2model')

from src.segmentation.predictor import FloorPlanPredictor
from src.segmentation.visualizer import SegmentationVisualizer

BEST_PT    = f'{results.save_dir}/weights/best.pt'
TEST_DIR   = '/kaggle/working/data/yolo_dataset/images/test'

# Pick the first test image
test_image = os.path.join(TEST_DIR, os.listdir(TEST_DIR)[0])
print(f'Testing on: {test_image}')

predictor = FloorPlanPredictor(BEST_PT)
viz       = SegmentationVisualizer()

result    = predictor.predict(test_image)
img       = cv2.imread(test_image)
annotated = viz.draw(img, result)

# Display inline in Kaggle
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(12, 8))
plt.imshow(annotated_rgb)
plt.axis('off')
plt.title(f'Detected {len(result.elements)} elements')
plt.tight_layout()
plt.savefig('/kaggle/working/test_prediction.png', dpi=150)
plt.show()
print('Prediction saved to: /kaggle/working/test_prediction.png')

In [ ]:
# ── Cell 9: Save outputs for download ─────────────────────────────────
# After this cell, go to the OUTPUT tab on the right panel
# Download best_final.pt to your Mac and place it at:
# floor2model/models/segmentation/best.pt
import shutil, os

BEST_PT = f'{results.save_dir}/weights/best.pt'

# Copy to /kaggle/working so it appears in the Output tab
shutil.copy(BEST_PT, '/kaggle/working/best_final.pt')
shutil.copy(
    f'{results.save_dir}/weights/last.pt',
    '/kaggle/working/last_final.pt'
)

size_mb = os.path.getsize('/kaggle/working/best_final.pt') / (1024*1024)
print(f'best_final.pt ready ({size_mb:.1f} MB)')
print(f'last_final.pt ready')
print()
print('Next steps:')
print('  1. Click the OUTPUT tab on the right panel')
print('  2. Download best_final.pt to your Mac')
print('  3. Place it at: floor2model/models/segmentation/best.pt')
print('  4. Come back to Claude for Phase 3!')